In [ ]:
import torch
from utils import *
from PIL import Image
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import torchvision.transforms.functional as TF

# ========================================
# CONFIGURATION
# ========================================
# Model checkpoint path
srgan_checkpoint = "./checkpoints/checkpoint_srgan_best.pth.tar"

# Set5 dataset paths
HR_folder = "data/benchmark/Set5/HR"
LR_folder = "data/benchmark/Set5/LR_bicubic/X4"

# Output directory for results
output_dir = "./results/Set5_SR_results"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, "SR_images"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "comparison"), exist_ok=True)

# Device configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("=" * 80)
print("BATCH SUPER-RESOLUTION FOR SET5 DATASET")
print("=" * 80)
print(f"Device: {device}")
print(f"Model: {srgan_checkpoint}")
print(f"HR Folder: {HR_folder}")
print(f"LR Folder: {LR_folder}")
print(f"Output Directory: {output_dir}")
print("=" * 80 + "\n")

# ========================================
# LOAD MODEL
# ========================================
print("Loading SRGAN model...")
checkpoint = torch.load(srgan_checkpoint, map_location=device, weights_only=False)
srgan_generator = checkpoint["generator"].to(device)
srgan_generator.eval()

# Check if EMA weights available (for better quality)
if "ema_shadow" in checkpoint:
    print("✓ EMA weights detected, applying for better quality...")
    from solver import EMA

    ema = EMA(srgan_generator)
    ema.shadow = checkpoint["ema_shadow"]
    ema.apply_shadow()

print("✓ Model loaded successfully\n")

# ========================================
# GET IMAGE LIST
# ========================================
print("Scanning Set5 dataset...")

# Get all HR images
hr_images = sorted(
    [f for f in os.listdir(HR_folder) if f.endswith((".png", ".jpg", ".bmp"))]
)

# Corresponding LR images (assuming naming convention: image.png -> imagex4.png)
lr_images = []
for hr_name in hr_images:
    base_name = os.path.splitext(hr_name)[0]
    lr_name = base_name + "x4.png"
    lr_images.append(lr_name)

print(f"✓ Found {len(hr_images)} image pairs\n")
print("Image list:")
for i, (hr, lr) in enumerate(zip(hr_images, lr_images), 1):
    print(f"  {i}. HR: {hr:<20} LR: {lr}")
print()

# ========================================
# PROCESS EACH IMAGE
# ========================================
print("=" * 80)
print("GENERATING SUPER-RESOLUTION IMAGES")
print("=" * 80 + "\n")

# Statistics for summary
total_images = len(hr_images)
success_count = 0
failed_images = []

with torch.no_grad():
    for idx, (hr_name, lr_name) in enumerate(
        tqdm(zip(hr_images, lr_images), total=total_images, desc="Processing")
    ):
        try:
            # Load images
            hr_path = os.path.join(HR_folder, hr_name)
            lr_path = os.path.join(LR_folder, lr_name)

            # Check if files exist
            if not os.path.exists(hr_path):
                print(f"⚠ Warning: HR image not found: {hr_path}")
                failed_images.append(hr_name)
                continue
            if not os.path.exists(lr_path):
                print(f"⚠ Warning: LR image not found: {lr_path}")
                failed_images.append(hr_name)
                continue

            # Load and convert images
            hr_img = Image.open(hr_path).convert("RGB")
            lr_img = Image.open(lr_path).convert("RGB")

            # Generate Bicubic upsampling (baseline)
            bicubic_img = lr_img.resize((hr_img.width, hr_img.height), Image.BICUBIC)

            # ========================================
            # Generate Super-Resolution with SRGAN
            # ========================================
            # Prepare LR tensor (on GPU)
            lr_tensor = convert_image(
                lr_img, source="pil", target="imagenet-norm", device=device
            )
            lr_tensor = lr_tensor.unsqueeze(0)

            # Forward pass (on GPU)
            sr_tensor = srgan_generator(lr_tensor)

            # Manual conversion: [-1, 1] GPU tensor -> PIL image
            # This avoids the device mismatch issue in convert_image
            sr_tensor = sr_tensor.squeeze(
                0
            ).cpu()  # Move to CPU, remove batch dimension
            sr_tensor = (sr_tensor + 1.0) / 2.0  # Denormalize from [-1, 1] to [0, 1]
            sr_tensor = torch.clamp(sr_tensor, 0.0, 1.0)  # Clamp to valid range

            # Convert to PIL using torchvision (guaranteed to work on CPU tensor)
            sr_img = TF.to_pil_image(sr_tensor)

            # ========================================
            # SAVE RESULTS
            # ========================================
            base_name = os.path.splitext(hr_name)[0]

            # 1. Save SR image only
            sr_save_path = os.path.join(output_dir, "SR_images", f"{base_name}_SR.png")
            sr_img.save(sr_save_path)

            # 2. Save comparison figure (LR, Bicubic, SR, HR)
            fig, axes = plt.subplots(2, 2, figsize=(12, 12))
            fig.suptitle(f"Comparison: {base_name}", fontsize=16, fontweight="bold")

            # LR image (upscaled for visualization)
            axes[0, 0].imshow(
                lr_img.resize((hr_img.width, hr_img.height), Image.NEAREST)
            )
            axes[0, 0].set_title(
                f"LR Input\n({lr_img.width}×{lr_img.height})", fontsize=12
            )
            axes[0, 0].axis("off")

            # Bicubic upsampling
            axes[0, 1].imshow(bicubic_img)
            axes[0, 1].set_title(
                f"Bicubic\n({bicubic_img.width}×{bicubic_img.height})", fontsize=12
            )
            axes[0, 1].axis("off")

            # SRGAN result
            axes[1, 0].imshow(sr_img)
            axes[1, 0].set_title(
                f"SRGAN\n({sr_img.width}×{sr_img.height})", fontsize=12
            )
            axes[1, 0].axis("off")

            # Original HR
            axes[1, 1].imshow(hr_img)
            axes[1, 1].set_title(
                f"Original HR\n({hr_img.width}×{hr_img.height})", fontsize=12
            )
            axes[1, 1].axis("off")

            plt.tight_layout()
            comparison_path = os.path.join(
                output_dir, "comparison", f"{base_name}_comparison.png"
            )
            plt.savefig(comparison_path, dpi=150, bbox_inches="tight")
            plt.close()

            # 3. Save side-by-side comparison (SR vs HR)
            fig, axes = plt.subplots(1, 2, figsize=(16, 8))
            fig.suptitle(
                f"SRGAN vs Ground Truth: {base_name}", fontsize=16, fontweight="bold"
            )

            axes[0].imshow(sr_img)
            axes[0].set_title("SRGAN Output", fontsize=14)
            axes[0].axis("off")

            axes[1].imshow(hr_img)
            axes[1].set_title("Ground Truth HR", fontsize=14)
            axes[1].axis("off")

            plt.tight_layout()
            sidebyside_path = os.path.join(
                output_dir, "comparison", f"{base_name}_sidebyside.png"
            )
            plt.savefig(sidebyside_path, dpi=150, bbox_inches="tight")
            plt.close()

            success_count += 1

        except Exception as e:
            print(f"\n✗ Error processing {hr_name}: {str(e)}")
            import traceback

            traceback.print_exc()
            failed_images.append(hr_name)
            continue

# ========================================
# GENERATE SUMMARY HTML
# ========================================
print("\n" + "=" * 80)
print("GENERATING SUMMARY HTML")
print("=" * 80 + "\n")

html_content = f"""
<!DOCTYPE html>
<html lang="en">
<head>
   <meta charset="UTF-8">
   <meta name="viewport" content="width=device-width, initial-scale=1.0">
   <title>Set5 Super-Resolution Results</title>
   <style>
       body {{
           font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
           margin: 0;
           padding: 20px;
           background-color: #f5f5f5;
       }}
       .header {{
           background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
           color: white;
           padding: 30px;
           border-radius: 10px;
           margin-bottom: 30px;
           box-shadow: 0 4px 6px rgba(0,0,0,0.1);
       }}
       h1 {{
           margin: 0;
           font-size: 2.5em;
       }}
       .stats {{
           display: grid;
           grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
           gap: 20px;
           margin-bottom: 30px;
       }}
       .stat-card {{
           background: white;
           padding: 20px;
           border-radius: 10px;
           box-shadow: 0 2px 4px rgba(0,0,0,0.1);
           text-align: center;
       }}
       .stat-number {{
           font-size: 2.5em;
           font-weight: bold;
           color: #667eea;
       }}
       .stat-label {{
           color: #666;
           margin-top: 10px;
       }}
       .gallery {{
           display: grid;
           grid-template-columns: repeat(auto-fill, minmax(400px, 1fr));
           gap: 20px;
       }}
       .image-card {{
           background: white;
           border-radius: 10px;
           overflow: hidden;
           box-shadow: 0 4px 6px rgba(0,0,0,0.1);
           transition: transform 0.3s;
       }}
       .image-card:hover {{
           transform: translateY(-5px);
           box-shadow: 0 6px 12px rgba(0,0,0,0.15);
       }}
       .image-card img {{
           width: 100%;
           height: auto;
           display: block;
       }}
       .image-title {{
           padding: 15px;
           font-weight: bold;
           color: #333;
           border-top: 3px solid #667eea;
       }}
       .download-btn {{
           background: #667eea;
           color: white;
           padding: 10px 20px;
           border-radius: 5px;
           text-decoration: none;
           display: inline-block;
           margin: 5px;
           transition: background 0.3s;
       }}
       .download-btn:hover {{
           background: #764ba2;
       }}
   </style>
</head>
<body>
   <div class="header">
       <h1>🎨 Set5 Super-Resolution Results</h1>
       <p>SRGAN 4× Super-Resolution Evaluation</p>
   </div>
   
   <div class="stats">
       <div class="stat-card">
           <div class="stat-number">{total_images}</div>
           <div class="stat-label">Total Images</div>
       </div>
       <div class="stat-card">
           <div class="stat-number">{success_count}</div>
           <div class="stat-label">Successfully Processed</div>
       </div>
       <div class="stat-card">
           <div class="stat-number">{len(failed_images)}</div>
           <div class="stat-label">Failed</div>
       </div>
       <div class="stat-card">
           <div class="stat-number">4×</div>
           <div class="stat-label">Upscaling Factor</div>
       </div>
   </div>
   
   <h2 style="margin-top: 40px; color: #333;">📊 Comparison Results</h2>
   <div class="gallery">
"""

# Add each image comparison to HTML
for hr_name in hr_images:
    base_name = os.path.splitext(hr_name)[0]
    comparison_rel_path = f"comparison/{base_name}_comparison.png"
    sidebyside_rel_path = f"comparison/{base_name}_sidebyside.png"
    sr_rel_path = f"SR_images/{base_name}_SR.png"

    if os.path.exists(os.path.join(output_dir, comparison_rel_path)):
        html_content += f"""
       <div class="image-card">
           <img src="{comparison_rel_path}" alt="{base_name} comparison">
           <div class="image-title">
               {base_name}
               <br>
               <a href="{sr_rel_path}" class="download-btn" download>Download SR</a>
               <a href="{sidebyside_rel_path}" class="download-btn">Side-by-Side</a>
           </div>
       </div>
       """

html_content += """
   </div>
   
   <h2 style="margin-top: 40px; color: #333;">📝 Notes</h2>
   <div style="background: white; padding: 20px; border-radius: 10px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
       <ul>
           <li><strong>LR Input:</strong> Low-resolution input image (bicubic downsampled)</li>
           <li><strong>Bicubic:</strong> Traditional bicubic interpolation upsampling (baseline)</li>
           <li><strong>SRGAN:</strong> Our super-resolution result using SRGAN</li>
           <li><strong>Original HR:</strong> Ground truth high-resolution image</li>
       </ul>
       <p><strong>Evaluation Tips:</strong></p>
       <ul>
           <li>Compare texture details and sharpness between SRGAN and Bicubic</li>
           <li>Check for artifacts or unnatural patterns in SRGAN results</li>
           <li>Evaluate color consistency with ground truth</li>
           <li>Assess perceptual quality vs pixel-wise accuracy</li>
       </ul>
   </div>
   
   <div style="margin-top: 40px; text-align: center; color: #666;">
       <p>Generated by SRGAN Super-Resolution Pipeline</p>
   </div>
</body>
</html>
"""

# Save HTML
html_path = os.path.join(output_dir, "index.html")
with open(html_path, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"✓ HTML summary saved: {html_path}")

# ========================================
# PRINT SUMMARY
# ========================================
print("\n" + "=" * 80)
print("PROCESSING COMPLETE")
print("=" * 80)
print(f"Total images: {total_images}")
print(f"Successfully processed: {success_count}")
print(f"Failed: {len(failed_images)}")
if failed_images:
    print(f"Failed images: {', '.join(failed_images)}")
print(f"\nResults saved to: {output_dir}")
print(f"  - SR images: {os.path.join(output_dir, 'SR_images')}")
print(f"  - Comparisons: {os.path.join(output_dir, 'comparison')}")
print(f"  - HTML summary: {html_path}")
print("\n💡 Open index.html in your browser to view all results!")
print("=" * 80 + "\n")